In [20]:
from openai import AsyncOpenAI
from typing import Dict, List, Optional
from datetime import datetime
import os
from pydantic import BaseModel
from dotenv import load_dotenv
from agents import Agent
import sqlite3
from agents import Agent
from openai import OpenAI
from agents import trace, Runner, OpenAIChatCompletionsModel

load_dotenv(override=True)


True

In [21]:
from pydantic import BaseModel
from typing import Optional, List

class Activity(BaseModel):
    type: str  # warmup | piece | improvisation | sight_reading

    # repertoire metadata
    piece_name: Optional[str] = None
    composer_name: Optional[str] = None
    key: Optional[str] = None
    section: Optional[str] = None
    bars: Optional[str] = None

    # warmup/exercise metadata
    exercise_name: Optional[str] = None
    tempo: Optional[int] = None
    repetitions: Optional[int] = None

    # general metadata
    focus: Optional[str] = None
    notes: Optional[str] = None


class PracticeSession(BaseModel):
    date: Optional[str]
    duration_min: Optional[int]
    raw_text: Optional[str]
    activities: List[Activity]


In [22]:
today = datetime.now().strftime("%Y-%m-%d")
PARSER_PROMPT = """
Extract structured piano practice data from the input text.

Return a JSON object with EXACTLY this structure:

{
  "date": string or null,
  "duration_min": integer or null,
  "raw_text": string,
  "activities": [
    {
      "type": "warmup" | "piece" | "improvisation" | "sight_reading",

      "piece_name": string or null,
      "composer_name": string or null,
      "key": string or null,
      "section": string or null,
      "bars": string or null,

      "exercise_name": string or null,
      "tempo": integer or null,
      "repetitions": integer or null,

      "focus": string or null,
      "notes": string or null
    }
  ]
}

Rules:
- If the date is missing, use today's date: """ + today + """.
- Duration must be an integer number of minutes. Convert formats like "~45 min", "50m", "1h20" into minutes.
- Split the text into multiple activities when appropriate.
- "type" must be one of: warmup, piece, improvisation, sight_reading.
- Do NOT invent activities or fields.
- If a field is not present in the text, set it to null.
- Keep all names and text exactly as written (no normalization).
- "raw_text" must contain the full original input.

Output rules:
- Return ONLY valid JSON.
- Do NOT wrap in markdown or code fences.
- Do NOT include explanations or comments.
- Output must start with "{" and end with "}".

"""

In [39]:
VALID_TYPES = ["warmup", "piece", "improvisation", "sight_reading"]
NORMALIZER_PROMPT = """
You MUST return ONLY valid JSON.

Rules:
- Output must start with "{" and end with "}".
- Do NOT use markdown.
- Do NOT use code fences.
- Do NOT add explanations, comments, or extra text.
- If you cannot produce valid JSON, return an empty JSON object: {}.

You are the Normalizer Agent.

Your job is to take a structured practice session JSON and normalize it so that
all names, types, keys, and piece references are consistent with the existing
database.

You have READ-ONLY access to the database through the provided tool. Use it
whenever needed to check existing composers, pieces, aliases, or previously
logged activities.

Normalization rules:
1. Enforce valid activity types:
   warmup, piece, improvisation, sight_reading

2. Normalize musical keys:
   - Capitalize correctly
   - Preserve accidentals (#, b)
   - Preserve "major" / "minor"

3. Normalize piece names:
   - If incomplete or misspelled, look up the correct name in the database.
   - If the composer name is incomplete, look it up in the database.
   - If the piece exists in the database, use the canonical_name stored there.

4. Normalize exercise names:
   - Use consistent naming for scales, arpeggios, octaves, etc.
   - If similar exercise names exist in the database, reuse them.

5. Do NOT remove information.
6. Do NOT invent new information.
7. If something cannot be normalized, keep the original value..

"""

## Validators

### Pipeline orchestation

### Agent definition

In [27]:
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AI
DeepSeek API Key exists and begins sk-


In [38]:
GEMINI_BASE_URL   = "https://generativelanguage.googleapis.com/v1beta/openai/"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"

deepseek_client   = AsyncOpenAI(base_url=DEEPSEEK_BASE_URL, api_key=deepseek_api_key)
gemini_client     = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

deepseek_model    = OpenAIChatCompletionsModel(model="deepseek-chat", openai_client=deepseek_client)
gemini_model      = OpenAIChatCompletionsModel(model="gemini-2.0-flash", openai_client=gemini_client)

In [ ]:
parserAgent = Agent(name="parser_Agent", instructions=PARSER_PROMPT, model=deepseek_model)
normAgent   = Agent(name="norm_Agent", instructions=NORMALIZER_PROMPT, model=gemini_model)
adv_agent3  = Agent(name="OAP_adv_Agent",instructions=PARSER_PROMPT,model="gpt-4o-mini")

In [ ]:
message = f"""15 apr 26, 50 min. Warmup: scales on D# minor and D major, weak fingers. 
Worked on Chopin Fantasie Impromptu op 66 in C# minor, bars 45-50 and 72-80, focusing on left hand balance, memory and speed. 
Improv on A minor."""


{
  "date": "2026-04-15",
  "duration_min": 50,
  "raw_text": "15 apr 26, 50 min. Warmup: scales on D# minor and D major, weak fingers. \nWorked on Chopin Fantasie Impromptu op 66 in C# minor, bars 45-50 and 72-80, focusing on left hand balance, memory and speed. \nImprov on A minor.",
  "activities": [
    {
      "type": "warmup",
      "piece_name": null,
      "composer_name": null,
      "key": "D# minor and D major",
      "section": null,
      "bars": null,
      "exercise_name": "scales",
      "tempo": null,
      "repetitions": null,
      "focus": "weak fingers",
      "notes": null
    },
    {
      "type": "piece",
      "piece_name": "Fantasie Impromptu op 66",
      "composer_name": "Chopin",
      "key": "C# minor",
      "section": null,
      "bars": "45-50 and 72-80",
      "exercise_name": null,
      "tempo": null,
      "repetitions": null,
      "focus": "left hand balance, memory and speed",
      "notes": null
    },
    {
      "type": "improvisation",
     

In [43]:
with trace("parser_Agent"):
    result = await Runner.run(parserAgent, message)
    print(result.final_output)

{
  "date": "2026-04-15",
  "duration_min": 50,
  "raw_text": "15 apr 26, 50 min. Warmup: scales on D# minor and D major, weak fingers. \nWorked on Chopin Fantasie Impromptu op 66 in C# minor, bars 45-50 and 72-80, focusing on left hand balance, memory and speed. \nImprov on A minor.",
  "activities": [
    {
      "type": "warmup",
      "piece_name": null,
      "composer_name": null,
      "key": "D# minor and D major",
      "section": null,
      "bars": null,
      "exercise_name": "scales",
      "tempo": null,
      "repetitions": null,
      "focus": "weak fingers",
      "notes": null
    },
    {
      "type": "piece",
      "piece_name": "Fantasie Impromptu op 66",
      "composer_name": "Chopin",
      "key": "C# minor",
      "section": null,
      "bars": "45-50 and 72-80",
      "exercise_name": null,
      "tempo": null,
      "repetitions": null,
      "focus": "left hand balance, memory and speed",
      "notes": null
    },
    {
      "type": "improvisation",
     

In [51]:
import json
from typing import Any

def extract_json(text: Any):
    """
    Robust JSON extractor:
    - If input is already a dict, return it.
    - If input is a valid JSON string, parse and return it.
    - Otherwise, extract the first JSON object using brace counting.
    """

    # Case 1: Already parsed JSON
    if isinstance(text, dict):
        return text

    # Case 2: Must be a string from the model
    if not isinstance(text, str):
        raise ValueError(f"Unsupported type for extract_json: {type(text)}")

    stripped = text.strip()

    # Case 3: Try direct JSON parse
    try:
        return json.loads(stripped)
    except json.JSONDecodeError:
        pass  # Not pure JSON, continue

    # Case 4: Extract JSON object using brace counting
    start = stripped.find("{")
    if start == -1:
        raise ValueError(f"No JSON object found in output:\n{text}")

    brace_count = 0
    in_json = False
    end = None

    for i, ch in enumerate(stripped[start:], start=start):
        if ch == "{":
            brace_count += 1
            in_json = True
        elif ch == "}":
            brace_count -= 1

        if in_json and brace_count == 0:
            end = i + 1
            break

    if end is None:
        raise ValueError(f"Unbalanced JSON braces in output:\n{text}")

    json_str = stripped[start:end]

    try:
        return json.loads(json_str)
    except json.JSONDecodeError as e:
        raise ValueError(f"Failed to parse extracted JSON:\n{json_str}\nError: {e}")




In [52]:
raw_output1 = result.final_output
parsed = extract_json(raw_output1)

In [48]:

with trace("Norm"):
    normResult = await Runner.run(
        normAgent,
        json.dumps(parsed)
    )

print(normResult.final_output)


```json
{"date": "2026-04-15", "duration_min": 50, "raw_text": "15 apr 26, 50 min. Warmup: scales on D# minor and D major, weak fingers. \nWorked on Chopin Fantasie Impromptu op 66 in C# minor, bars 45-50 and 72-80, focusing on left hand balance, memory and speed. \nImprov on A minor.", "activities": [{"type": "warmup", "piece_name": null, "composer_name": null, "key": "D# minor and D major", "section": null, "bars": null, "exercise_name": "scales", "tempo": null, "repetitions": null, "focus": "weak fingers", "notes": null}, {"type": "piece", "piece_name": "Fantasie Impromptu op 66", "composer_name": "Chopin", "key": "C# minor", "section": null, "bars": "45-50 and 72-80", "exercise_name": null, "tempo": null, "repetitions": null, "focus": "left hand balance, memory and speed", "notes": null}, {"type": "improvisation", "piece_name": null, "composer_name": null, "key": "A minor", "section": null, "bars": null, "exercise_name": null, "tempo": null, "repetitions": null, "focus": null, "not

In [53]:
raw_output2 = normResult.final_output
normJSON = extract_json(raw_output2)
print(normJSON)


{'date': '2026-04-15', 'duration_min': 50, 'raw_text': '15 apr 26, 50 min. Warmup: scales on D# minor and D major, weak fingers. \nWorked on Chopin Fantasie Impromptu op 66 in C# minor, bars 45-50 and 72-80, focusing on left hand balance, memory and speed. \nImprov on A minor.', 'activities': [{'type': 'warmup', 'piece_name': None, 'composer_name': None, 'key': 'D# minor and D major', 'section': None, 'bars': None, 'exercise_name': 'scales', 'tempo': None, 'repetitions': None, 'focus': 'weak fingers', 'notes': None}, {'type': 'piece', 'piece_name': 'Fantasie Impromptu op 66', 'composer_name': 'Chopin', 'key': 'C# minor', 'section': None, 'bars': '45-50 and 72-80', 'exercise_name': None, 'tempo': None, 'repetitions': None, 'focus': 'left hand balance, memory and speed', 'notes': None}, {'type': 'improvisation', 'piece_name': None, 'composer_name': None, 'key': 'A minor', 'section': None, 'bars': None, 'exercise_name': None, 'tempo': None, 'repetitions': None, 'focus': None, 'notes': Non

In [54]:
import sqlite3

conn = sqlite3.connect("piano.db")
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")


In [55]:
def normalize(text):
    return text.lower().strip() if text else None

def get_or_create_composer(name):
    if not name:
        return None
    norm = normalize(name)
    cursor.execute("SELECT id FROM composers WHERE normalized_name = ?", (norm,))
    row = cursor.fetchone()
    if row:
        return row[0]
    cursor.execute(
        "INSERT INTO composers (name, normalized_name) VALUES (?, ?)",
        (name, norm)
    )
    return cursor.lastrowid

def get_or_create_piece(piece_name, composer_id):
    if not piece_name:
        return None
    norm = normalize(piece_name)
    cursor.execute("""
        SELECT id FROM pieces
        WHERE normalized_name = ? AND composer_id IS ?
    """, (norm, composer_id))
    row = cursor.fetchone()
    if row:
        return row[0]
    cursor.execute("""
        INSERT INTO pieces (canonical_name, normalized_name, composer_id)
        VALUES (?, ?, ?)
    """, (piece_name, norm, composer_id))
    return cursor.lastrowid


In [56]:
def log_session(session):
    cursor.execute(
        "INSERT INTO sessions (date, duration_min, raw_text) VALUES (?, ?, ?)",
        (session["date"], session["duration_min"], session["raw_text"])
    )
    session_id = cursor.lastrowid

    for act in session["activities"]:
        composer_id = get_or_create_composer(act.get("composer_name"))
        piece_id = get_or_create_piece(act.get("piece_name"), composer_id)

        cursor.execute("""
            INSERT INTO activities (
                session_id, type,
                piece_id, composer_id,
                piece_name, composer_name,
                key, section, bars,
                exercise_name, tempo, repetitions,
                focus, notes
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            session_id,
            act.get("type"),

            piece_id,
            composer_id,

            act.get("piece_name"),
            act.get("composer_name"),

            act.get("key"),
            act.get("section"),
            act.get("bars"),

            act.get("exercise_name"),
            act.get("tempo"),
            act.get("repetitions"),

            act.get("focus"),
            act.get("notes"),
        ))

    conn.commit()


In [57]:
log_session(normJSON)
print("Inserted normalized session into piano.db")

Inserted normalized session into piano.db
